In [16]:
IDtoLabel_UK = {
    0: 'O',
    1: 'B-Person',
    2: 'I-Person',
    3: 'B-Organization',
    4: 'I-Organization',
    5: 'B-Location',
    6: 'I-Location',
    7: 'B-Misc',
    8: 'I-Misc'
}
IDtoLabel_US = {
    0:'O',
    1:'B-Misc',
    2:'B-Date',
    3:'I-Date',
    4:'B-Person',
    5:'I-Person',
    6:'B-NORP',
    7:'B-Misc',
    8:'I-Misc',
    9:'B-Misc',
    10:'I-Misc',
    11:'B-Organization',
    12:'I-Organization',
    13:'B-Misc',
    14:'I-Misc',
    15:'B-Misc',
    16:'B-Money',
    17:'I-Money',
    18:'B-Misc',
    19:'I-Misc',
    20:'B-Facility',
    21:'B-Date',
    22:'I-Misc',
    23:'B-Location',
    24:'B-Misc',
    25:'I-Misc',
    26:'I-NORP',
    27:'I-Location',
    28:'B-Product',
    29:'I-Date',
    30:'B-Misc',
    31:'I-Misc',
    32:'I-Facility',
    33:'B-Misc',
    34:'I-Product',
    35:'I-Misc',
    36:'I-Misc'
}
LabelToID = {
    'O': 0,
    'B-Date': 1,
    'I-Date': 2,
    'B-Person': 3,
    'I-Person': 4,
    'B-Location': 5,
    'I-Location': 6,
    'B-Facility': 7,
    'I-Facility': 8,
    'B-Organization': 9,
    'I-Organization': 10,
    'B-Misc': 11,
    'I-Misc': 12,
    'B-Money': 13,
    'I-Money': 14,
    'B-NORP': 15,
    'I-NORP': 16,
    'B-Product': 17,
    'I-Product': 18
}

In [18]:
from datasets import load_dataset
data=load_dataset("BramVanroy/conll2003")

# Extract the list of string labels to map integers back to text
# Standard CoNLL-2003 tags: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']
trainDataUK=data['train'].to_pandas().drop(columns=['pos_tags', 'chunk_tags', 'id'])
trainDataUK['ner_tags'] = trainDataUK['ner_tags'].apply(lambda x: [IDtoLabel_UK[i] for i in x])
trainDataUK['ner_tags'] = trainDataUK['ner_tags'].apply(lambda x: [LabelToID[i] for i in x])
valDataUK=data['validation'].to_pandas().drop(columns=['pos_tags', 'chunk_tags', 'id'])
valDataUK['ner_tags'] = valDataUK['ner_tags'].apply(lambda x: [IDtoLabel_UK[i] for i in x])
valDataUK['ner_tags'] = valDataUK['ner_tags'].apply(lambda x: [LabelToID[i] for i in x])
testDataUK=data['test'].to_pandas().drop(columns=['pos_tags', 'chunk_tags', 'id'])
testDataUK['ner_tags'] = testDataUK['ner_tags'].apply(lambda x: [IDtoLabel_UK[i] for i in x])
testDataUK['ner_tags'] = testDataUK['ner_tags'].apply(lambda x: [LabelToID[i] for i in x])

In [ ]:
"""One file only"""
import os
import glob
import pandas as pd

def load_custom_tsv(file_path):
    sentences=[]
    ner_tags=[]

    current_sentence=[]
    current_tags=[]

    with open(file_path, 'r', encoding='utf-8') as f:
        try:            # Skip the first two lines (the link and the empty line)
            next(f)
            next(f)
        except StopIteration:
            pass # Failsafe just in case the file is completely empty

        for line in f:
            line=line.strip()

            if not line:    #empty line aka end of sentence
                if current_sentence:
                    sentences.append(list(current_sentence))
                    ner_tags.append(list(current_tags))
                    current_sentence.clear()
                    current_tags.clear()
                continue
            parts=line.split('\t')

            if len(parts) >= 2:
                token=parts[0]
                tag=parts[-1]

                current_sentence.append(token)
                current_tags.append(tag)

        if current_sentence:
            sentences.append(list(current_sentence))
            ner_tags.append(list(current_tags))

    df = pd.DataFrame({'tokens': sentences, 'ner_tags': ner_tags})
    
    return df

file_path = r"C:\Users\anna-\OneDrive\Desktop\ITU\4th semester\NLP and Deep Learning\af_afrol_1.txt.tsv"
dataGlobal = load_custom_tsv(file_path)
dataGlobal

In [20]:
import os
import glob
import pandas as pd

def load_tsvs_from_folder(folder_path):
    all_sentences = []
    all_ner_tags = []
    all_regions = []

    search_pattern = os.path.join(folder_path, "*.tsv")
    file_list = glob.glob(search_pattern)
    
    for file_path in file_list:
        filename = os.path.basename(file_path)
        region_name = filename.split('_')[0]
        current_sentence = []
        current_tags = []

        with open(file_path, 'r', encoding='utf-8') as f:
            for _ in range(2):       # Skip the first two lines
                try:
                    next(f)
                except StopIteration:
                    break

            for line in f:
                line = line.strip()

                if not line:    # EMPTY LINE: Sentence is over
                    if current_sentence:
                        all_sentences.append(list(current_sentence))
                        all_ner_tags.append(list(current_tags))
                        all_regions.append(region_name)
                        current_sentence.clear()
                        current_tags.clear()
                    continue 
                    
                parts = line.split('\t')    # TEXT: Extract token and primary tag
                if len(parts) >= 2:
                    token = parts[0].strip()
                    raw_tag = parts[1].strip() # Grab the FIRST tag to avoid nested tags

                    current_sentence.append(token)
                    current_tags.append(raw_tag)

            if current_sentence:        # END OF FILE: Catch the final sentence
                all_sentences.append(list(current_sentence))
                all_ner_tags.append(list(current_tags))
                all_regions.append(region_name)

    df = pd.DataFrame({'tokens': all_sentences, 'ner_tags': all_ner_tags, 'region': all_regions,})
    return df

folder_path = r".\processed_annotated"
dataGlobal = load_tsvs_from_folder(folder_path)

dataGlobal

,tokens,ner_tags,region
0,"[DURBAN, ,, May, 20, 2022, (, IPS, ), -, Gover...","[B-Location, O, B-Date, I-Date, I-Date, O, B-O...",afs
1,"[These, were, among, the, diverse, opinions, o...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",afs
2,"[Hundreds, of, delegates, ,, including, world,...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",afs
3,"[Sessions, and, panel, discussions, highlighte...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",afs
4,"[Speaking, during, the, closing, ceremony, on,...","[O, O, O, O, O, O, B-Date, O, B-Location, I-Lo...",afs
...,...,...,...
21312,"[Fanny, is, about, to, give, a, picture, of, h...","[B-Person, O, O, O, O, O, O, O, O, O, O, O, O,...",ven
21313,"[But, the, majority, of, the, civil, society, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",ven
21314,"[It, ’s, all, about, a, perpetual, and, useful...","[O, O, O, O, O, O, O, O, O, O, O, O, B-Person,...",ven
21315,"[Lusverti, adds, :, “, The, struggle, of, thes...","[B-Person, O, O, O, O, O, O, O, O, O, O, O, O,...",ven


In [21]:
dataGlobal['ner_tags'] = dataGlobal['ner_tags'].apply(lambda x: [LabelToID[i] for i in x])

In [22]:
dataGlobal

,tokens,ner_tags,region
0,"[DURBAN, ,, May, 20, 2022, (, IPS, ), -, Gover...","[5, 0, 1, 2, 2, 0, 9, 0, 0, 0, 0, 0, 0, 0, 0, ...",afs
1,"[These, were, among, the, diverse, opinions, o...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",afs
2,"[Hundreds, of, delegates, ,, including, world,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",afs
3,"[Sessions, and, panel, discussions, highlighte...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",afs
4,"[Speaking, during, the, closing, ceremony, on,...","[0, 0, 0, 0, 0, 0, 1, 0, 5, 6, 6, 6, 0, 0, 0, ...",afs
...,...,...,...
21312,"[Fanny, is, about, to, give, a, picture, of, h...","[3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",ven
21313,"[But, the, majority, of, the, civil, society, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",ven
21314,"[It, ’s, all, about, a, perpetual, and, useful...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, ...",ven
21315,"[Lusverti, adds, :, “, The, struggle, of, thes...","[3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",ven


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import pandas as pd
import numpy as np

def build_vocab(datasets_list): # Vocabulary mapping words to integer IDs
    word2idx = {"<PAD>": 0, "<UNK>": 1}
    for df in datasets_list:
        for tokens in df['tokens']:
            for word in tokens:
                if word not in word2idx:
                    word2idx[word] = len(word2idx)
    return word2idx

class NERDataset(Dataset):      # PyTorch Dataset Class
    def __init__(self, df, word2idx):
        self.sentences = df['tokens'].tolist()
        self.labels = df['ner_tags'].tolist()
        self.word2idx = word2idx

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):     # Convert words to indices, fallback to <UNK> if word is unseen
        word_indices = [self.word2idx.get(w, self.word2idx["<UNK>"]) for w in self.sentences[idx]]
        label_indices = self.labels[idx]
        return torch.tensor(word_indices), torch.tensor(label_indices)

def collate_fn(batch):      # function to pad sentences to the same length in a batch
    sentences, labels = zip(*batch)
    padded_sentences = pad_sequence(sentences, batch_first=True, padding_value=0)   # Pad sentences with 0 (<PAD>)
    padded_labels = pad_sequence(labels, batch_first=True, padding_value=-100)      # Pad labels with -100 (PyTorch CrossEntropyLoss ignores -100 by default)
    return padded_sentences, padded_labels

In [ ]:
class RNN_NER(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes):
        super(RNN_NER, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)     # Word embedding layer
        
        self.lstm = nn.LSTM(            # Bidirectional LSTM
            embedding_dim, 
            hidden_dim, 
            batch_first=True, 
            bidirectional=True)
        
        # Linear layer mapping hidden states to tag classes
        self.fc = nn.Linear(hidden_dim * 2, num_classes)        # hidden_dim * 2 because it's bidirectional

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, _ = self.lstm(embedded)
        logits = self.fc(lstm_out)
        return logits

In [ ]:
def train_model(model, dataloader, optimizer, criterion, device, epochs=3):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for sentences, labels in dataloader:
            sentences, labels = sentences.to(device), labels.to(device)
            
            optimizer.zero_grad()
            logits = model(sentences)
            
            # Reshape logits and labels for CrossEntropyLoss
            logits = logits.view(-1, logits.shape[-1])      # logits: (batch_size * seq_len, num_classes)
            labels = labels.view(-1)        # labels: (batch_size * seq_len)
            
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(dataloader):.4f}")

def evaluate_model(model, dataloader, device):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for sentences, labels in dataloader:
            sentences, labels = sentences.to(device), labels.to(device)
            logits = model(sentences)
            
            preds = torch.argmax(logits, dim=-1)        # Get the predicted class indices
            
            preds = preds.view(-1).cpu().numpy()        # Flatten to compare
            labels = labels.view(-1).cpu().numpy()
            
            mask = labels != -100           # Filter out the padded tokens (-100)
            all_preds.extend(preds[mask])
            all_labels.extend(labels[mask])
            
    f1 = f1_score(all_labels, all_preds, average='macro')
    return f1

In [27]:
# --- 1. PREPARE YOUR DATAFRAMES ---
# Assuming you have `df_english` (CoNLL) and `df_worldwide` (dataGlobal)
# First, split the Worldwide dataset since CoNLL already has predefined splits
train_ww, test_ww = train_test_split(dataGlobal, test_size=0.2, random_state=42)

# Create Combined datasets
train_combined = pd.concat([trainDataUK, train_ww], ignore_index=True)

# Build a master vocabulary from ALL training data
word2idx = build_vocab([trainDataUK, train_ww])
vocab_size = len(word2idx)
num_classes = len(LabelToID) # 19 classes based on your notebook

# --- 2. DEFINE THE EXPERIMENT MATRIX ---
experiments = [
    {"name": "Train: Worldwide | Test: Worldwide", "train": train_ww, "test": test_ww},
    {"name": "Train: English   | Test: Worldwide", "train": trainDataUK, "test": test_ww},
    {"name": "Train: Worldwide | Test: English",   "train": train_ww,  "test": testDataUK},
    {"name": "Train: English   | Test: English",   "train": trainDataUK, "test": testDataUK},
    {"name": "Train: Combined  | Test: Worldwide", "train": train_combined, "test": test_ww},
    {"name": "Train: Combined  | Test: English",   "train": train_combined, "test": testDataUK},
]

# --- 3. RUN THE EXPERIMENTS ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

results = {}

for exp in experiments:
    print(f"\n{'='*50}\nRunning: {exp['name']}\n{'='*50}")
    
    # Create DataLoaders
    train_ds = NERDataset(exp["train"], word2idx)
    test_ds = NERDataset(exp["test"], word2idx)
    
    train_dl = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate_fn)
    test_dl = DataLoader(test_ds, batch_size=32, shuffle=False, collate_fn=collate_fn)
    
    # Initialize a fresh model for each experiment
    model = RNN_NER(vocab_size, embedding_dim=128, hidden_dim=256, num_classes=num_classes).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss(ignore_index=-100) # Ignores padding!
    
    # Train
    train_model(model, train_dl, optimizer, criterion, device, epochs=5)
    
    # Evaluate
    f1_score_val = evaluate_model(model, test_dl, device)
    print(f"Result -> F1 Score: {f1_score_val:.4f}")
    
    results[exp['name']] = f1_score_val

# Final Summary
print("\n--- Final Experiment Results (F1 Score) ---")
for name, score in results.items():
    print(f"{name}: {score:.4f}")


Running: Train: Worldwide | Test: Worldwide
Epoch 1/5 | Loss: 0.4619
Epoch 2/5 | Loss: 0.2137
Epoch 3/5 | Loss: 0.1315
Epoch 4/5 | Loss: 0.0800
Epoch 5/5 | Loss: 0.0462
Result -> F1 Score: 0.7035

Running: Train: English   | Test: Worldwide
Epoch 1/5 | Loss: 0.5438
Epoch 2/5 | Loss: 0.2202
Epoch 3/5 | Loss: 0.1166
Epoch 4/5 | Loss: 0.0585
Epoch 5/5 | Loss: 0.0254
Result -> F1 Score: 0.1901

Running: Train: Worldwide | Test: English
Epoch 1/5 | Loss: 0.4650
Epoch 2/5 | Loss: 0.2148
Epoch 3/5 | Loss: 0.1324
Epoch 4/5 | Loss: 0.0818
Epoch 5/5 | Loss: 0.0480
Result -> F1 Score: 0.2140

Running: Train: English   | Test: English
Epoch 1/5 | Loss: 0.5439
Epoch 2/5 | Loss: 0.2233
Epoch 3/5 | Loss: 0.1168
Epoch 4/5 | Loss: 0.0568
Epoch 5/5 | Loss: 0.0241
Result -> F1 Score: 0.7166

Running: Train: Combined  | Test: Worldwide
Epoch 1/5 | Loss: 0.4177
Epoch 2/5 | Loss: 0.1846
Epoch 3/5 | Loss: 0.1063
Epoch 4/5 | Loss: 0.0587
Epoch 5/5 | Loss: 0.0299
Result -> F1 Score: 0.6857

Running: Train: Co